# Zhang et al. (2017), *Understanding deep learning requires rethinking generalization* — Figure 1

Reproduction of Figure 1 (a) learning curves of the small Inception net on CIFAR-10 under the five randomization tests,
(b) convergence slowdown vs. label corruption, (c) test error vs. label corruption, for Inception, AlexNet and MLP 1x512.

All code is in this folder (`data.py`, `models.py`, `train.py`, `plot_fig1.py`). `run_all.sh` runs the whole sweep
(about 3 h on an Apple M4 Pro GPU). Set `RUN_TRAINING = True` below to retrain; otherwise the saved logs in `results/` are used.

In [ ]:
import os, sys, json, glob, subprocess
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
RUN_TRAINING = False          # True: run the full sweep (hours). False: use the logs in results/
if RUN_TRAINING:
    subprocess.run(["bash", "run_all.sh"], check=True)

## Sanity checks: parameter counts match Table 1 of the paper

In [ ]:
from models import build, count_params
for a, ref in [("inception", 1_649_402), ("alexnet", 1_387_786), ("mlp1", 1_209_866), ("mlp3", 1_735_178)]:
    print(f"{a:10s} {count_params(build(a)):>10,}  paper {ref:>10,}")

## Sanity checks: the randomization tests
With corruption probability $p$ a label is replaced by a uniform draw from the 10 classes, so only $0.9p$ of the labels actually change.

In [ ]:
import torch
from data import load_cifar10, preprocess, randomize
xtr, ytr, xte, yte = load_cifar10(os.environ.get("CIFAR10_NPZ", "data/cifar10.npz"))
xtr_t, xte_t = preprocess(xtr[:2000]), preprocess(xte[:500])
ytr_t, yte_t = torch.from_numpy(ytr[:2000]).long(), torch.from_numpy(yte[:500]).long()
print("per-image whitening: mean %.3f std %.3f" % (xtr_t[0].mean(), xtr_t[0].std()))
for p in [0.2, 0.5, 1.0]:
    _, yc, _, _ = randomize(xtr_t, ytr_t, xte_t, yte_t, "corrupt", p)
    print(f"p={p}: fraction of labels changed = {(yc != ytr_t).float().mean():.3f}  (expected {0.9*p:.3f})")
fig, ax = plt.subplots(1, 5, figsize=(12, 2.6))
for i, mode in enumerate(["true", "corrupt", "shuffled_pixels", "random_pixels", "gaussian"]):
    xs, _, _, _ = randomize(xtr_t, ytr_t, xte_t, yte_t, mode, 1.0)
    img = xs[0].permute(1, 2, 0).numpy(); img = (img - img.min()) / (img.max() - img.min())
    ax[i].imshow(img); ax[i].set_title(mode); ax[i].axis("off")
plt.show()

## Figure 1 reproduced
`plot_fig1.py` draws the three panels from the JSON logs. Convergence time ("time to overfit") is the number of SGD
steps until the full-training-set accuracy first reaches the threshold (default 99%; see the report for why not 100%).

In [ ]:
!python3 plot_fig1.py --results results --out figures/fig1_reproduced.png
display(Image("figures/fig1_reproduced.png"))

## Per-run summary table

In [ ]:
rows = []
for f in sorted(glob.glob("results/*.json")):
    d = json.load(open(f))
    rows.append((d["tag"], d["epoch"][-1], d["train_acc"][-1], 1 - d["test_acc"][-1], d.get("steps_to_100"), d.get("frac_labels_changed")))
print(f"{'run':32s} {'epochs':>6s} {'train_acc':>9s} {'test_err':>8s} {'steps_to_100':>12s} {'frac_changed':>12s}")
for r in rows:
    print(f"{r[0]:32s} {r[1]:6d} {r[2]:9.4f} {r[3]:8.4f} {str(r[4]):>12s} {str(None if r[5] is None else round(r[5],3)):>12s}")